# Task 3 -- Define Input Schema and Validate

Loads `data/interim/cleaned.csv` (Task 2's output), validates every row against the schema
defined below, and routes rows to `data/interim/validated.csv` or
`data/interim/rejected.csv` (with the failure reason logged).


In [ ]:
import json
import math

import pandas as pd

cleaned_df = pd.read_csv("../data/interim/cleaned.csv")
print("Loaded", cleaned_df.shape, "from ../data/interim/cleaned.csv")

# JSON Schema validation needs real dicts, not a DataFrame -- NaN -> None.
records = cleaned_df.where(pd.notnull(cleaned_df), None).to_dict(orient="records")
print(f"{len(records)} rows to validate")


## Schema Definition

| Column | Data Type | Nullable | Allowed Values / Range |
|---|---|---|---|
| `external_id` | string | N | must match `devto:<digits>` |
| `title` | string | N | non-empty |
| `source` | string | N | must equal `dev.to` |
| `author` | string | N | non-empty |
| `published_date` | string (ISO datetime) | N | valid ISO 8601 timestamp |
| `url` | string | N | must start with `https://` |
| `topic` | string | Y | any string, empty allowed |
| `tags` | string | Y | comma-separated tag names, empty allowed |
| `description` | string | Y | free text, empty allowed |
| `content_type` | string | N | must equal `article` |
| `reading_time_minutes` | integer | N | >= 0 |
| `reactions_count` | integer | N | >= 0 |
| `comments_count` | integer | N | >= 0 |
| `cover_image` | string (URL) | Y | starts with `http` when present |
| `content_markdown` | string | Y | raw article body, empty allowed if the content fetch failed |
| `content_html` | string | Y | raw article body (HTML), empty allowed if the content fetch failed |
| `content_clean` | string | Y | plain-text body, empty allowed if the content fetch failed |


In [ ]:
SCHEMA = {
    "type": "object",
    "required": [
        "external_id", "title", "source", "author", "published_date", "url",
        "content_type", "reading_time_minutes", "reactions_count", "comments_count",
    ],
    "properties": {
        "external_id": {"type": "string", "pattern": r"^devto:\d+$"},
        "title": {"type": "string", "minLength": 1},
        "source": {"type": "string", "enum": ["dev.to"]},
        "author": {"type": "string", "minLength": 1},
        "published_date": {"type": "string", "minLength": 1},
        "url": {"type": "string", "pattern": r"^https://"},
        "topic": {"type": ["string", "null"]},
        "tags": {"type": ["string", "null"]},
        "description": {"type": ["string", "null"]},
        "content_type": {"type": "string", "enum": ["article"]},
        "reading_time_minutes": {"type": "number", "minimum": 0},
        "reactions_count": {"type": "number", "minimum": 0},
        "comments_count": {"type": "number", "minimum": 0},
        "cover_image": {"type": ["string", "null"]},
        "content_markdown": {"type": ["string", "null"]},
        "content_html": {"type": ["string", "null"]},
        "content_clean": {"type": ["string", "null"]},
    },
}
print("Schema has", len(SCHEMA["properties"]), "columns defined,", len(SCHEMA["required"]), "required")


## Validate: route each row to `validated` or `rejected` (with a logged reason)

In [ ]:
from jsonschema import Draft7Validator

validator = Draft7Validator(SCHEMA)

validated_rows = []
rejected_rows = []

for record in records:
    errors = sorted(validator.iter_errors(record), key=lambda e: e.path)
    if not errors:
        validated_rows.append(record)
    else:
        reason = "; ".join(f"{'.'.join(map(str, e.path)) or '(row)'}: {e.message}" for e in errors)
        rejected_record = dict(record)
        rejected_record["rejection_reason"] = reason
        rejected_rows.append(rejected_record)

print(f"Validated: {len(validated_rows)} rows")
print(f"Rejected:  {len(rejected_rows)} rows")

if rejected_rows:
    print("\nSample rejection reasons:")
    for r in rejected_rows[:5]:
        print(" -", r["rejection_reason"])


## Save `validated.csv` and `rejected.csv`

In [ ]:
import os

INTERIM_DIR = os.path.join("..", "data", "interim")
os.makedirs(INTERIM_DIR, exist_ok=True)

validated_path = os.path.join(INTERIM_DIR, "validated.csv")
rejected_path = os.path.join(INTERIM_DIR, "rejected.csv")

pd.DataFrame(validated_rows).to_csv(validated_path, index=False)
pd.DataFrame(rejected_rows).to_csv(rejected_path, index=False)

print(f"Saved -> {validated_path} ({len(validated_rows)} rows)")
print(f"Saved -> {rejected_path} ({len(rejected_rows)} rows)")
